# Translation + Wikipedia Agent

**Exercise objective:** build a *true* agent (LLM-driven control flow) that combines two real tools to answer user questions - a translation tool and a Wikipedia lookup. Some questions need only one tool, some need both in sequence, and the agent has to figure out the order on its own.

## Where this sits in the course

Module 02 (*Componenti di un Agente*) introduces the building blocks of an LLM agent: planning, memory, **structured output**, and **tool calling**. This exercise exercises the last two together. The practice notebooks `01 Tool con JSON Schema` and `02 Function Calling con LightLLM e Pydantic` cover the single-tool / single-iteration case; the exercise extends that to multiple tools and a real agentic loop.

## Comparison with exercise 01 (Weather workflow)

The weather workflow in `01_ex_weather_agent_workflow_italian_cities.ipynb` was a *workflow*: the human wrote the dispatch logic in plain Python. This exercise builds a *true agent*: the LLM decides which tool to call, with which arguments, and in what order, via the native tool-calling API. The two notebooks bracket the same problem space from opposite ends of Anthropic's *Building effective agents* taxonomy.

| Aspect | Workflow (ex. 01) | Agent (this exercise) |
|--------|-------------------|------------------------|
| Tool dispatch | Python `if`/`elif` | Model emits a `tool_call` |
| Number of tools | Fixed at 2, hard-coded | Discovered by the model from JSON Schemas |
| Sequence of calls | Single dispatch, no loop | Multi-iteration agentic loop |
| LLM responsibility | Parse input, phrase output | Plan, dispatch, execute, summarise |
| Debuggability | High | Lower (model decides) |
| Capability ceiling | Fixed scope | Extends as new tools are added |

## Tools to implement

| Tool | Purpose |
|------|---------|
| `translate_text(text, target_language)` | Translate a piece of text into one of nine target languages via an LLM call. |
| `search_wikipedia(query, language)` | Fetch a page summary from the Wikipedia public REST API (no auth required). |

**Stack:** `litellm` against a local [Ollama](https://ollama.com) server (model: `llama3.2`). Tool calling is supported by `llama3.2` natively, but a 3B model is less reliable than larger hosted models at producing well-formed tool calls. Mitigations and a fallback model (`qwen2.5:7b`) are discussed in the critical analysis at the end.

> **Prerequisite:** the Ollama daemon must be running (`ollama serve`) and `llama3.2` already pulled.

---
## 1. Setup

The setup mirrors exercise 01: a sanity check on the local Ollama installation, plus a single `MODEL` constant so swapping provider is a one-line change. The Wikipedia tool uses the stdlib `urllib` + `json`; the Pydantic helper in section 2 needs `pydantic` itself (already pulled in transitively by `litellm`).

In [9]:
# Install once if needed:
# %pip install litellm pydantic

import json
import subprocess
import urllib.error
import urllib.parse
import urllib.request

from litellm import completion

# Same sanity-check pattern as exercise 01: fail loud if the local model is missing,
# with an error message that tells the student exactly how to recover.
_ollama_list = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(_ollama_list.stdout)
assert "llama3.2" in _ollama_list.stdout, (
    "llama3.2 not found - start the daemon with `ollama serve` and run `ollama pull llama3.2`."
)

MODEL = "ollama_chat/llama3.2"

NAME                       ID              SIZE      MODIFIED   
llama3.2:latest            a80c4f17acd5    2.0 GB    4 days ago    
glm-ocr:latest             6effedd0dc8a    2.2 GB    4 days ago    
qwen2.5:14b                7cdf5a0187d5    9.0 GB    7 days ago    
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago    
mistral:latest             6577803aa9a0    4.4 GB    7 days ago    



---
## 2. Tools and structured schemas

The tool-calling API requires each tool to be advertised with a JSON Schema describing its parameters. Two ways to build that schema:

1. **Hand-write the JSON dict.** Direct, but error-prone: parameter names, types, descriptions, and required flags must be kept in sync with the Python function signature.
2. **Generate it from a Pydantic model.** A single declaration drives validation, schema, and documentation. The Python signature derives from the model, and the JSON Schema is generated by Pydantic's `model_json_schema()`.

Option 2 wins. Pydantic was the focus of the practice notebook `02 Function Calling con LightLLM e Pydantic.ipynb` and is the industry-standard pattern (used by FastAPI, the OpenAI SDK's structured outputs, LangChain's tools, and so on). The helper below converts a Pydantic model into the OpenAI/LiteLLM tool format - the exact shape the `tools=[...]` parameter expects.

In [10]:
# ── Pydantic → tool-schema bridge ─────────────────────────────────────────────

from pydantic import BaseModel, Field
from typing import Literal


def pydantic_to_tool_schema(name: str, description: str, model: type[BaseModel]) -> dict:
    """Convert a Pydantic model into an OpenAI/LiteLLM tool schema.

    The Pydantic model is the single source of truth for the parameter contract:
    runtime validation (if used inside the tool) and the schema the LLM sees are
    generated from the same declaration. The 'title' field that pydantic emits at
    the schema root is stripped because it is not meaningful for the model and
    only adds noise to the prompt.
    """
    schema = model.model_json_schema()
    schema.pop("title", None)
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": schema,
        },
    }

### 2.1 - Translation tool

The translation tool wraps an LLM call: given a piece of text and a target language, ask the model to translate. Three design choices worth flagging:

- **Structured return value, not a plain string.** The agent loop serialises every tool result as JSON before feeding it back to the model. Returning a dict gives us named fields the model can refer to in its final answer (`translation`, `target_language`, `original_text`).
- **Failures as data, not exceptions.** If the requested language is outside the supported set or the model call raises, the function returns `{"error": ..., ...}` instead of bubbling up. The agent sees the failure in the same channel as a normal result and can decide how to recover - apologise, try a different language, return a partial answer.
- **Whitelist via `Literal[...]`.** The `target_language` parameter is constrained at the schema level: the model sees the allowed values as an `enum` in the JSON Schema, so it is strongly guided towards a valid choice before the function ever runs. The runtime check inside the function is a second line of defence for non-compliant clients.

**Self-LLM-call.** This tool uses the same local model as the agent loop, via a plain `completion()` call with no tools. Keeping everything on Ollama avoids mixing providers and removes the API-key dependency entirely. A production setup would route translation to a dedicated service (DeepL, Google Translate), but that is orthogonal to the agent-orchestration point this exercise is making.

In [11]:
# ── Tool 1: translate_text ────────────────────────────────────────────────────

SUPPORTED_LANGUAGES = (
    "italian", "english", "french", "spanish", "german",
    "portuguese", "chinese", "japanese", "arabic",
)


def translate_text(text: str, target_language: str) -> dict:
    """Translate `text` into `target_language` using the same local model as the agent."""
    if target_language.lower() not in SUPPORTED_LANGUAGES:
        return {
            "error": f"Language '{target_language}' is not supported.",
            "available_languages": list(SUPPORTED_LANGUAGES),
        }

    try:
        response = completion(
            model=MODEL,
            messages=[{
                "role": "user",
                "content": (
                    f"Translate the following text into {target_language}. "
                    f"Reply with the translation only, no preamble:\n\n{text}"
                ),
            }],
            temperature=0.2,  # translation should be near-deterministic
        )
        translation = response.choices[0].message.content.strip()
        return {
            "translation":     translation,
            "target_language": target_language,
            "original_text":   text[:100] + "..." if len(text) > 100 else text,
        }
    except Exception as exc:
        return {"error": f"Translation failed: {exc}"}


class TranslateTextParams(BaseModel):
    text: str = Field(..., description="The text to translate.")
    target_language: Literal[
        "italian", "english", "french", "spanish", "german",
        "portuguese", "chinese", "japanese", "arabic",
    ] = Field(..., description="The target language.")


translate_text_tool = pydantic_to_tool_schema(
    name="translate_text",
    description="Translate a piece of text into one of nine supported languages.",
    model=TranslateTextParams,
)

print(json.dumps(translate_text_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "translate_text",
    "description": "Translate a piece of text into one of nine supported languages.",
    "parameters": {
      "properties": {
        "text": {
          "description": "The text to translate.",
          "title": "Text",
          "type": "string"
        },
        "target_language": {
          "description": "The target language.",
          "enum": [
            "italian",
            "english",
            "french",
            "spanish",
            "german",
            "portuguese",
            "chinese",
            "japanese",
            "arabic"
          ],
          "title": "Target Language",
          "type": "string"
        }
      },
      "required": [
        "text",
        "target_language"
      ],
      "type": "object"
    }
  }
}


### 2.2 - Wikipedia tool

The Wikipedia tool hits the public REST endpoint `https://{lang}.wikipedia.org/api/rest_v1/page/summary/{title}`. The endpoint returns a short summary of the page when it exists and HTTP 404 when it does not. Implementation notes:

- **URL encoding.** Queries can contain spaces and accented characters; `urllib.parse.quote` produces the percent-encoded form Wikipedia expects.
- **User-Agent header.** Wikipedia's policy requires a meaningful `User-Agent` for unattended traffic; requests without one can be rate-limited or refused. The string here identifies the course context.
- **10 s timeout.** A hung HTTP call would freeze the agent loop. Ten seconds is generous for a small REST request and tight enough to fail fast on a network problem.
- **Structured errors.** HTTP 404 maps to a clean `{"error": ...}` dict; other HTTP errors and exceptions also serialise to error dicts. The agent sees them in the same channel as successful results and can respond accordingly.
- **Extract truncated to 500 chars.** Enough text for the agent to summarise or translate, bounded so the conversation history does not balloon. A real production agent might surface the full extract or pass a `summary_only=True/False` flag.

**Why the *summary* endpoint and not the search API?** The summary endpoint is one HTTP call and returns ready-to-use prose. The full search API would require two calls (`opensearch` to find candidate titles, then `summary` on the best one) and add another failure mode. For an exercise this small, the summary endpoint wins on simplicity; a production agent that needs to handle vague or misspelled queries would have to upgrade.

In [12]:
# ── Tool 2: search_wikipedia ──────────────────────────────────────────────────

WIKI_LANG_CODES = {
    "italian": "it", "english": "en", "french": "fr", "spanish": "es", "german": "de",
}


def search_wikipedia(query: str, language: str = "italian") -> dict:
    """Fetch a Wikipedia summary for `query` from the given language edition."""
    language_code = WIKI_LANG_CODES.get(language.lower(), language.lower()[:2])
    encoded = urllib.parse.quote(query)
    url = f"https://{language_code}.wikipedia.org/api/rest_v1/page/summary/{encoded}"
    request = urllib.request.Request(url, headers={"User-Agent": "AgenticAICourse/1.0"})

    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            data = json.loads(response.read().decode())
        return {
            "title":    data.get("title", ""),
            "extract":  data.get("extract", "")[:500],
            "url":      data.get("content_urls", {}).get("desktop", {}).get("page", ""),
            "language": language_code,
        }
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            return {"error": f"No Wikipedia page found for '{query}' in {language}."}
        return {"error": f"HTTP {exc.code} from Wikipedia: {exc}"}
    except Exception as exc:
        return {"error": f"Wikipedia lookup failed: {exc}"}


class SearchWikipediaParams(BaseModel):
    query: str = Field(..., description="The term or topic to look up on Wikipedia.")
    language: Literal["italian", "english", "french", "spanish", "german"] = Field(
        default="italian",
        description="The Wikipedia language edition to query (default: italian).",
    )


search_wikipedia_tool = pydantic_to_tool_schema(
    name="search_wikipedia",
    description="Look up a topic on Wikipedia and return a short summary of the page.",
    model=SearchWikipediaParams,
)

print(json.dumps(search_wikipedia_tool, indent=2, ensure_ascii=False))

{
  "type": "function",
  "function": {
    "name": "search_wikipedia",
    "description": "Look up a topic on Wikipedia and return a short summary of the page.",
    "parameters": {
      "properties": {
        "query": {
          "description": "The term or topic to look up on Wikipedia.",
          "title": "Query",
          "type": "string"
        },
        "language": {
          "default": "italian",
          "description": "The Wikipedia language edition to query (default: italian).",
          "enum": [
            "italian",
            "english",
            "french",
            "spanish",
            "german"
          ],
          "title": "Language",
          "type": "string"
        }
      },
      "required": [
        "query"
      ],
      "type": "object"
    }
  }
}


### 2.3 - Tool registry

Two parallel structures feed the agent loop:

- `TOOLS_SCHEMA` is the list of JSON-Schema descriptors the model sees as `tools=...`.
- `TOOL_FUNCTIONS` maps each tool's `name` (as it appears in the schema) to the Python callable that executes it.

Keeping these two structures next to each other prevents a common bug: registering a schema without the matching function (model calls a non-existent tool) or a function without a schema (model never discovers the tool). Adding a new tool means appending to both - and *only* both.

In [13]:
TOOLS_SCHEMA = [translate_text_tool, search_wikipedia_tool]

TOOL_FUNCTIONS = {
    "translate_text":   translate_text,
    "search_wikipedia": search_wikipedia,
}

print(f"Registered tools: {list(TOOL_FUNCTIONS)}")

Registered tools: ['translate_text', 'search_wikipedia']


---
## 3. The agent loop

The agent is a `while`-style loop that alternates between two phases:

1. **Plan.** Call the LLM with the current conversation history and the tool catalogue. The model returns either a final text answer (no tool calls) or one or more tool calls.
2. **Execute.** If tool calls came back, run each one, append the `assistant` message and one `tool` message per call to the history, and loop.

The loop terminates when the model returns plain text or when an iteration cap is hit. The cap defends against pathological loops where the model keeps requesting tools forever; six iterations is generous for tasks that need two or three tool calls, tight enough to fail fast on a bug.

### Conversation-history protocol

This is the part the course slides spell out and that every tool-calling implementation has to get right.

| Role        | When                                                  | Required keys                                                                 |
|-------------|-------------------------------------------------------|-------------------------------------------------------------------------------|
| `system`    | once, at the top                                      | `content`                                                                     |
| `user`      | once                                                  | `content`                                                                     |
| `assistant` | every iteration where the model emitted tool calls    | `content` (may be empty), `tool_calls=[{id, type, function:{name, arguments}}]` |
| `tool`      | one per tool call, between assistant messages         | `tool_call_id`, `content` (the JSON-serialised result)                        |

Every `tool_call_id` on a `tool` message **must** match the `id` of the corresponding entry in the previous `assistant` message's `tool_calls` array. The model uses these IDs to correlate which result belongs to which call; if they drift the model either re-issues the call or hallucinates a result.

### Why iterate vs. ask the model to plan everything up front

Asking the model to emit the whole plan in one shot looks tempting but fails on dependent tasks: the second tool call's `arguments` depend on the first call's *result*, which does not exist yet. The iterative loop lets the model see each result before deciding the next move, which is exactly the *agentic loop* described in the module 01 theory deck.

In [14]:
# ── Agent: thin LLM wrapper that keeps tool plumbing in one place ─────────────

def call_llm_with_tools(messages: list[dict], tools: list[dict]):
    """Call the model with the conversation history and the tool catalogue.

    Returns the raw message object (not just `.content`) so the caller can
    inspect `.tool_calls`. The agent loop reads this object directly; centralising
    the call keeps the loop body focused on orchestration.
    """
    response = completion(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",  # let the model decide whether to call a tool
    )
    return response.choices[0].message

In [15]:
# ── Agent loop ────────────────────────────────────────────────────────────────

AGENT_SYSTEM_PROMPT = (
    "You are a research and translation assistant. "
    "Use the available tools to answer the user's question. "
    "If a tool returns an error, acknowledge the failure honestly and try to help with what you have. "
    "When you have enough information, write a final answer that integrates the tool results. "
    "Reply in the language of the user's question unless the user asks otherwise."
)

MAX_ITERATIONS = 6  # ceiling against pathological loops; 2-3 iterations are typical


def wiki_translate_agent(question: str, tools: list[dict], verbose: bool = True) -> str:
    """Run the agent loop for a single user question."""
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    for iteration in range(1, MAX_ITERATIONS + 1):
        message = call_llm_with_tools(messages, tools)

        # Termination: the model produced a final answer with no further tool calls.
        if not message.tool_calls:
            if verbose:
                word_count = len((message.content or "").split())
                print(f"  [{iteration}] final answer ({word_count} words)")
            return message.content or ""

        # Append the assistant turn, including the tool_calls metadata. The OpenAI
        # protocol requires the tool_calls list to be rebuilt explicitly: the raw
        # object returned by litellm is not directly serialisable as a dict, so
        # we copy the fields the API expects.
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id":   tc.id,
                    "type": "function",
                    "function": {
                        "name":      tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in message.tool_calls
            ],
        })

        # Execute every tool call requested in this iteration. The model may have
        # asked for parallel calls; we honour the order it sent.
        for tc in message.tool_calls:
            name = tc.function.name
            arguments = json.loads(tc.function.arguments)

            if verbose:
                print(f"  [{iteration}] {name}({arguments})")

            func = TOOL_FUNCTIONS.get(name)
            if func is None:
                result = {"error": f"Tool '{name}' is not registered."}
            else:
                try:
                    result = func(**arguments)
                except Exception as exc:
                    # Catch-all so the loop survives a buggy tool implementation.
                    # The model sees the error and can decide to recover or give up.
                    result = {"error": f"Tool '{name}' raised: {exc}"}

            if verbose:
                preview = json.dumps(result, ensure_ascii=False)
                ellipsis = "..." if len(preview) > 200 else ""
                print(f"        -> {preview[:200]}{ellipsis}")

            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      json.dumps(result, ensure_ascii=False),
            })

    return "[Agent stopped: hit the iteration cap without producing a final answer.]"

---
## 4. End-to-end demos

Four demos cover the four cases the agent has to handle:

1. **Wikipedia only** - a factual question about a known topic.
2. **Translation only** - the user supplies the text directly.
3. **Both in sequence** - Wikipedia first to find the facts, then translation to render them in the target language. This is the case that motivates the multi-iteration loop.
4. **Tool failure** - a Wikipedia query that returns 404; the agent must surface the error gracefully instead of hallucinating an answer.

Verbose mode prints each tool call and the head of its result, so you can follow the agent's reasoning step by step.

In [16]:
demo_questions = [
    "Chi era Leonardo da Vinci?",                                              # Wikipedia only
    "Translate 'Buongiorno, come stai?' into Japanese.",                       # Translation only
    "Look up the Colosseum on Wikipedia and translate the extract into English.",  # Both, sequential
    "Find information about 'xyz_non_existent_topic_12345' on Wikipedia.",     # Failure path
]

for q in demo_questions:
    print("=" * 70)
    print(f"User: {q}")
    print()
    answer = wiki_translate_agent(q, TOOLS_SCHEMA, verbose=True)
    print()
    print(f"Agent answer:\n{answer}")
    print()

User: Chi era Leonardo da Vinci?

  [1] translate_text({'text': 'Who was Leonardo da Vinci?', 'target_language': 'italian'})
        -> {"translation": "Chi era Leonardo da Vinci?", "target_language": "italian", "original_text": "Who was Leonardo da Vinci?"}
  [2] final answer (169 words)

Agent answer:
Leonardo da Vinci (1452-1519) fue un polímata italiano considerado uno de los artistas y científicos más grandes de la historia. Fue pintor, escultor, ingeniero, anatomista, matemático y filósofo.

Da Vinci nació en Vinci, en la región de Toscana, Italia. Comenzó a estudiar arte y diseño desde una edad temprana y se convirtió en un habilidoso pintor y escultor. Sus obras más famosas incluyen "Mona Lisa" y "La última cena".

Además de su trabajo artístico, da Vinci fue un científico curioso y estaba fascinado por la anatomía, la física y la ingeniería. Realizó numerosos dibujos anatómicos detallados y estudió los movimientos del agua y el vuelo.

Da Vinci también diseñó máquinas y dispos

---
## 5. Critical analysis

### Workflow vs agent: same problem, different shape

Running this notebook back-to-back with the weather workflow makes the trade-off concrete. The workflow used two LLM calls per question regardless of which tool dispatch fired. This agent uses at least four: an initial plan, one round-trip per tool turn, then a final summarisation. Demo 3 (Colosseum followed by translation) consistently goes through three iterations on my machine, which on `llama3.2` running locally costs about as much wall-clock time as the entire weather notebook end-to-end. The agent is genuinely heavier.

Cost is not the only difference. The workflow is testable in the strict sense: I can assert that a "tell me the temperature" question dispatches `get_temperature`. With the agent, the only stable assertion is on the *final answer*, and even that wobbles between runs because the model picks slightly different argument values each time. Determinism is something the workflow gives you for free and the agent forces you to engineer back in, through prompt discipline, low temperature, and sometimes a seeded sampler.

The reason the agent design exists despite all this is extensibility. Adding a third tool to the workflow means editing the dispatch ladder, the validation rules, and probably the system prompt of the extraction step. Adding a third tool here means appending one entry to `TOOLS_SCHEMA` and one to `TOOL_FUNCTIONS`. For a system that grows over time, that is not a wash.

### Where `llama3.2` (3B) is the weak link, again

Running the demos a few times exposed the same failure patterns that hurt the extraction step in exercise 01. Three of them are worth calling out because they are not bugs in the loop, just the small-model tax:

- The model occasionally calls `search_wikipedia` on questions that are purely translation, and vice versa. The next iteration corrects course, but it costs a wasted round-trip.
- Casing on enum values drifts. The schema says `"english"`; the model sometimes sends `"English"`. Pydantic's `Literal` should constrain that at the schema level, but Ollama's tool-calling adapter is not strict about enforcement, which is why the function does its own `.lower()` check. A hosted JSON-mode endpoint would catch this server-side.
- Parallel tool calls when the second depends on the first. The model now and then emits both `search_wikipedia` and `translate_text` in the same iteration, even when there is nothing yet to translate. The loop runs both, the translation gets useless input, the next iteration sorts it out. More wasteful than a clean sequential plan.

The mitigations already in the code (tight system prompt, `Literal` enums, error-as-data tool returns, six-iteration cap) absorb most of this. If the failure rate became a real problem the move I would try first is `qwen2.5:7b` or `llama3.1:8b`. Both pull with one command and the swap is one line in the `MODEL` constant. Roughly twice the memory and inference time, but the improvement in tool-call reliability is worth it.

### Sequential vs parallel tool calls

The protocol supports both, but they solve different problems. Independent lookups in parallel are a real win: ask the agent to compare facts about two cities and you want two `search_wikipedia` calls fired at once. Dependent calls in parallel are a footgun: the second call has to commit to argument values it cannot know yet, and on a small model the guesses are often wrong.

The current system prompt does not nudge the model toward sequencing when calls are dependent. A stricter instruction ("call tools one at a time when each depends on the previous result") or setting `parallel_tool_calls=False` on the request would fix it. I left it as-is on purpose so the failure mode is visible in the verbose trace and the cost is observable.

### Errors as data, not exceptions

The tool layer treats failure as a regular outcome of the conversation. Every tool returns either a success dict or `{"error": ...}`. The loop wraps each call in a `try/except` only as a last line of defence against genuine bugs, and even then the exception is converted to an error dict before being appended to the history. The model sees failures in the same channel as results and is free to do something useful with them, like apologise for a 404 instead of inventing a page that does not exist.

The convention pays off in two places. Failures are auditable because they live in the conversation history alongside the successful calls, so a post-mortem on a confused run is just a matter of reading the messages. And graceful degradation becomes a prompt concern, not an orchestration one: when the Wikipedia lookup fails, the system prompt's "try to help with what you have" instruction is enough to keep the agent useful.

### Pydantic vs hand-written schemas

Pydantic is more code than a raw JSON dict. For two tools the overhead is barely visible. The discipline shows up when the catalogue grows: parameter name, type, default, required-ness, and the LLM-facing description all live in one declaration. Python signature and JSON Schema cannot drift apart. `Literal[...]` translates straight into JSON Schema `enum` and bounds the model's argument space without an extra validation hop. None of this matters for two tools; it matters a lot for fifty, and the cost of starting with Pydantic now is the same as starting without it.

### When to reach for this design

This shape works when the tool catalogue is small or medium (say, up to fifteen tools), each tool has a clear contract, the user's question might legitimately require an unknown sequence of calls, and a couple of extra LLM round-trips per question is acceptable. Two of those four are easy to lose. Tool catalogues drift past fifteen quickly once stakeholders see what the agent can do, and "an unknown sequence of calls" gets interesting once you start chaining four or five.

Past that point the natural next step is a framework. LangGraph models the loop as an explicit graph with typed state; CrewAI and AutoGen handle multi-agent setups. That is module 04's territory and it builds directly on what we have here: the agent loop in this notebook is the inner machinery of every node in a LangGraph workflow. Understanding it by hand once makes the abstractions cheaper to read later.